# Customer Churn Analysis

Exploratory data analysis and machine learning pipeline to predict customer churn using the IBM Telco dataset.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import DATA_PATH, build_preprocessor, load_data, prepare_features

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load Data

In [ ]:
df = load_data(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 2. Data Overview

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
missing = df.isnull().sum()
missing[missing > 0]

## 3. Churn Distribution

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_rate = (df["Churn"] == "Yes").mean()
print(f"Churn rate: {churn_rate:.1%}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Churn", ax=ax[0], palette="Set2")
ax[0].set_title("Churn Count")
ax[1].pie(churn_counts, labels=churn_counts.index, autopct="%1.1f%%", colors=sns.color_palette("Set2"))
ax[1].set_title("Churn Proportion")
plt.tight_layout()

## 4. Churn by Key Features

In [ ]:
def plot_churn_rate(column: str) -> None:
    rate = df.groupby(column)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)
    plt.figure(figsize=(8, 4))
    sns.barplot(x=rate.index, y=rate.values, palette="Reds_r")
    plt.title(f"Churn Rate by {column}")
    plt.ylabel("Churn Rate")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

for feature in ["Contract", "InternetService", "PaymentMethod", "SeniorCitizen"]:
    plot_churn_rate(feature)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[0], palette="Set2")
axes[0].set_title("Tenure by Churn Status")
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1], palette="Set2")
axes[1].set_title("Monthly Charges by Churn Status")
plt.tight_layout()

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
corr_df = df[numeric_cols + ["SeniorCitizen"]].copy()
corr_df["Churn"] = (df["Churn"] == "Yes").astype(int)

plt.figure(figsize=(6, 4))
sns.heatmap(corr_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.tight_layout()

## 5. Prepare Features

In [ ]:
X, y = prepare_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")

## 6. Train and Compare Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

results = {}
fitted_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", estimator),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fitted_models[name] = pipeline
    results[name] = {
        "roc_auc": roc_auc_score(y_test, y_proba),
        "report": classification_report(y_test, y_pred),
        "y_pred": y_pred,
        "y_proba": y_proba,
    }
    print(f"\n{name}")
    print(results[name]["report"])

In [ ]:
comparison = pd.DataFrame(
    {name: {"ROC AUC": data["roc_auc"]} for name, data in results.items()}
).T.sort_values("ROC AUC", ascending=False)
comparison

## 7. Model Evaluation Plots

In [ ]:
best_name = comparison.index[0]
best_model = fitted_models[best_name]
y_pred = results[best_name]["y_pred"]
y_proba = results[best_name]["y_proba"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(ax=axes[0], cmap="Blues")
axes[0].set_title(f"Confusion Matrix - {best_name}")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title(f"ROC Curve - {best_name}")
plt.tight_layout()

## 8. Key Insights

- **Contract type** is a strong predictor: month-to-month customers churn at much higher rates.
- **Tenure** inversely correlates with churn — newer customers are more likely to leave.
- **Fiber optic internet** subscribers show elevated churn, often linked to pricing and support issues.
- **Electronic check** payment method correlates with higher churn vs automatic payments.

Run `python src/train.py` to persist the best model, then launch the dashboard with `streamlit run app/streamlit_app.py`.